# CareLoop Clinics — Part 1
Tasks 1.1–1.6. Run this notebook from the repository root.

In [ ]:
import csv, json, requests
import pandas as pd
import numpy as np

APPT_PATH = "data/appointments.csv"
CLINIC_PATH = "data/clinics.csv"


## Task 1.1 — csv module

In [ ]:
def parse_row(row):
    row = row.copy()
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

accepted = []
rejected = []

with open(APPT_PATH, newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    rows = list(reader)
    print("Row count:", len(rows))

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)


## Task 1.2 — inspect and clean pass 1

In [ ]:
df = pd.read_csv(APPT_PATH)
print("Shape:", df.shape)
df.info()
print("\nNulls:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())

df = df.drop_duplicates()
print("After dropping duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Distinct booking channels after cleaning:", df["booking_channel"].nunique())

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Unreadable/missing appointment dates:", df["appt_date"].isna().sum())
print("Missing statuses:", df["status"].isna().sum())

df = df.dropna(subset=["appt_date", "status"]).copy()
print("After dropping missing appt_date/status:", len(df))


## Task 1.3 — validity, gaps and outliers

In [ ]:
invalid_age = ~df["age"].between(1, 100)
print("Invalid ages:", int(invalid_age.sum()))
df.loc[invalid_age, "age"] = np.nan

age_median = df["age"].median()
distance_median = df["distance_km"].median()
print("Age median:", age_median)
print("Distance median:", distance_median)

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    capped = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: Q1={q1}, Q3={q3}, lower={lower}, upper={upper}, capped={capped}")
    df[col] = np.clip(df[col], lower, upper)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)


## Task 1.4 — features and NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("Column means:", np.mean(X, axis=0))

mask = df["lead_days"].to_numpy() > 14
print("lead_days > 14:", int(mask.sum()), "share:", mask.mean())

distance = X[:, 1]
z = (distance - np.mean(distance)) / np.std(distance)
print("|z| > 2:", int((np.abs(z) > 2).sum()))

print("No-shows:", int(df["no_show"].sum()), "rate:", round(df["no_show"].mean()*100, 1), "%")


## Task 1.5 — Open-Meteo enrichment

In [ ]:
clinics = pd.read_csv(CLINIC_PATH)

def get_temp(row):
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }
    last_reason = None
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )
            if response.status_code != 200:
                last_reason = f"HTTP {response.status_code}"
                continue
            data = response.json()
            hourly = data.get("hourly")
            if not hourly or "temperature_2m" not in hourly:
                last_reason = "hourly.temperature_2m missing"
                continue
            return hourly["temperature_2m"][0]
        except requests.exceptions.RequestException as exc:
            last_reason = str(exc)
    print(f"{row['clinic_name']}: {last_reason}")
    return None

clinics["temp_c"] = clinics.apply(get_temp, axis=1)
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)


## Task 1.6 — export clean table and SQL

In [ ]:
clean_cols = [
    "appointment_id","patient_id","clinic_id","booked_on","appt_date","age",
    "gender","distance_km","fee_inr","booking_channel","visit_type",
    "reminder_sent","prior_no_shows","status","lead_days","no_show"
]

out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[clean_cols]

print("appointment_id unique:", out["appointment_id"].is_unique)
print("Missing values across 16 columns:", int(out[clean_cols].isna().sum().sum()))
out.to_csv("appointments_clean.csv", index=False)

def sql_value(value, numeric=False):
    if pd.isna(value):
        return "NULL"
    if numeric:
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"

with open("load_data.sql", "w") as f:
    for _, r in pd.read_csv("clinics_enriched.csv").iterrows():
        vals = [
            sql_value(r["clinic_id"]), sql_value(r["clinic_name"]),
            sql_value(r["city"]), sql_value(r["tier"]),
            sql_value(r["latitude"], True), sql_value(r["longitude"], True),
            sql_value(r["temp_c"], True)
        ]
        f.write(f"INSERT INTO clinics VALUES ({','.join(vals)});\n")

    for _, r in out.iterrows():
        vals = [
            sql_value(r["appointment_id"]), sql_value(r["patient_id"]),
            sql_value(r["clinic_id"]), sql_value(r["booked_on"]),
            sql_value(r["appt_date"]), sql_value(r["age"], True),
            sql_value(r["gender"]), sql_value(r["distance_km"], True),
            sql_value(r["fee_inr"], True), sql_value(r["booking_channel"]),
            sql_value(r["visit_type"]), sql_value(r["reminder_sent"]),
            sql_value(r["prior_no_shows"], True), sql_value(r["status"]),
            sql_value(r["lead_days"], True), sql_value(r["no_show"], True)
        ]
        f.write(f"INSERT INTO appointments VALUES ({','.join(vals)});")

print("Rows:", len(out), "SQL INSERTs:", 8 + len(out))
